# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Notebook hoàn chỉnh cho E0–E4. Mã dùng chung nằm trong `student_vlm.py`; chạy các ô theo thứ tự và chỉ dùng số liệu thật sinh ra trong `report/evidence/`.

## 0. Môi trường (Kaggle/Colab GPU 16 GB)

In [ ]:
# Kaggle: Settings → Accelerator → GPU T4 x1 (hoặc P100). Colab: chọn T4.
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Không hard-code secret. Nếu đã Add Data trên Kaggle thì không cần API key.
import json
import os
from pathlib import Path
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except ImportError:
    pass
print('CUDA runtime ready; secrets are read only when a Kaggle CLI download is needed.')

## 1. Dữ liệu: tải và chia split

In [ ]:
# Trên Kaggle nên Add Data: lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes.
import subprocess, sys
KAGGLE_INPUT = Path('/kaggle/input/road-damage-dataset-potholes-cracks-and-manholes')
prepare_cmd = [sys.executable, 'scripts/prepare_data.py']
prepare_cmd += ['--source', str(KAGGLE_INPUT)] if KAGGLE_INPUT.exists() else ['--kaggle']
subprocess.run(prepare_cmd, check=True)

In [ ]:
from student_vlm import read_jsonl, show_examples, split_summary
train_rows = read_jsonl('data/splits/train.jsonl')
val_rows = read_jsonl('data/splits/val.jsonl')
test_rows = read_jsonl('data/splits/test.jsonl')
stats = split_summary()  # đồng thời assert không rò session giữa các split
print(json.dumps(stats, ensure_ascii=False, indent=2))
show_examples(train_rows, n=6)

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
from student_vlm import objects_to_answer, yolo_to_box
sample = train_rows[0]
answer = objects_to_answer(sample['objects'])
print(answer)
assert answer == objects_to_answer(sample['objects'])  # thứ tự/JSON ổn định
assert all(0 <= v <= 1000 for o in json.loads(answer) for v in o['bbox_2d'])

## 3. E0: zero-shot

In [ ]:
import gc
import torch
from student_vlm import load_base_model, predict
RUN_E0 = True
if RUN_E0:
    e0_model, e0_processor = load_base_model(training=False)
    predict(e0_model, e0_processor, test_rows, 'work/e0.jsonl', batch_size=4)
    del e0_model, e0_processor
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
if RUN_E0:
    subprocess.run([sys.executable, 'scripts/score.py', '--pred', 'work/e0.jsonl',
                    '--name', 'E0', '--out', 'report/evidence/e0.json'], check=True)

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
from student_vlm import VlmCollator, encode_example
# Chạy kiểm tra này sau khi processor đã được tạo ở bước 5. Hàm chỉ tính loss trên answer.
def check_encoding(processor):
    encoded = encode_example(processor, train_rows[0])
    answer_ids = encoded['labels'][encoded['labels'] != -100]
    decoded = processor.tokenizer.decode(answer_ids, skip_special_tokens=False)
    image_pad_id = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
    vision_tokens = int((encoded['input_ids'] == image_pad_id).sum())
    print('vision tokens:', vision_tokens, 'answer:', decoded)
    assert objects_to_answer(train_rows[0]['objects']) in decoded
    return encoded

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
from student_vlm import apply_setup, load_base_model
SETUP = 'E1'       # chạy lại các bước 5–7 lần lượt với E1, E2, E3
RANK = 16
if 'model' in globals():
    del model, processor
    gc.collect(); torch.cuda.empty_cache()
model, processor = load_base_model(training=True)
check_encoding(processor)
model, setup_info = apply_setup(model, SETUP, rank=RANK)
print(setup_info)
assert setup_info['matched_modules'] == (348 if SETUP == 'E3' else 252)

## 6. Vòng huấn luyện

In [ ]:
from student_vlm import train
RUN_DIR = f'work/{SETUP.lower()}'
run_info = train(
    model, processor, train_rows, RUN_DIR, setup_info=setup_info,
    lr=2e-4, batch_size=2, grad_accum=4, max_steps=300, seed=42,
)
print(json.dumps(run_info, indent=2))

## 7. Suy luận và chấm E1–E3

In [ ]:
import subprocess
from student_vlm import predict, show_predictions
pred_path = f'work/{SETUP.lower()}.jsonl'
predict(model, processor, test_rows, pred_path, batch_size=4)
cmd = [
    sys.executable, 'scripts/score.py', '--pred', pred_path, '--name', SETUP,
    '--out', f'report/evidence/{SETUP.lower()}.json',
    '--meta', f"trainable_params={run_info['trainable_params']}",
    '--meta', f"peak_vram_gb={run_info['peak_vram_gb']}",
    '--meta', f"train_minutes={run_info['train_minutes']}",
]
subprocess.run(cmd, check=True)
Path('report/figures').mkdir(parents=True, exist_ok=True)
fig = show_predictions(test_rows, pred_path, n=6)
fig.savefig(f'report/figures/{SETUP.lower()}_examples.png', dpi=160, bbox_inches='tight')
del model, processor
gc.collect(); torch.cuda.empty_cache()
# Đổi SETUP và chạy lại bước 5–7 cho thí nghiệm tiếp theo.

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# E4 rank ablation: chỉ chạy sau khi đã có e1/e2/e3.json.
# Tự chọn setup có F1 cao nhất, rồi huấn luyện rank 4 và 64; alpha=2*rank.
evidence = {s: json.loads(Path(f'report/evidence/{s}.json').read_text()) for s in ('e1','e2','e3')}
BEST_SETUP = max(evidence, key=lambda s: evidence[s]['micro']['f1']).upper()
print('Best base setup:', BEST_SETUP)
RUN_E4 = False  # đổi thành True sau khi xác nhận BEST_SETUP
if RUN_E4:
    from student_vlm import run_sft_experiment
    for rank in (4, 64):
        result = run_sft_experiment(
            name=f'E4_rank_{rank}', setup=BEST_SETUP, rank=rank,
            train_records=train_rows, test_records=test_rows,
        )
        print(rank, result['micro']['f1'], result['f1_ci95'])
# E5 là bonus và không được chạy nếu chưa viết/test reward riêng theo GUIDE bước 9.
!python scripts/check_submission.py